# 🚀 Build Your First AI Agent: GDG Study Jam Edition

**Welcome to the Google Developer Group Study Jam!**

In the next **~75 minutes**, you'll build an AI agent that can **take actions** to solve problems. No prior AI experience needed!

## 🎯 What You'll Learn

- ✅ What is an AI Agent? (Spoiler Alert: it's cooler than ChatGPT!)
- ✅ Build your first agent with **Google's Agent Development Kit (ADK)**
- ✅ Create **custom tools** to let your agent interact with the world.
- ✅ **Multi-tool agents**: Teach agents to choose the right tool for the job
- ✅ **Sessions & memory**: Give agents the ability to remember context
- ✅ **Multi-agent systems**: Use agents as tools for other agents

## ⏱️ Pace (~75 minutes) (TBD)

- **Setup & Concepts**: 15 min
- **Learn & Build Together**: 35 min (tools, multi-tool agents, sessions)
- **You Customize & Explore**: 15 min
- **Show & Tell**: 5 min
- **Wrap Up & Resources**: 5 min

---

## ⚙️ Setup: Get Your Environment Ready

**Before we start coding, let's set up your development environment.**

You'll need:
- Python 3.12+
- ADK library
- A Google AI Studio API key

### Create your API key (5 quick steps)
1. Open https://aistudio.google.com/app/api-keys
2. Click **Get API key** (or **Create API key**) in the top-right
3. Choose or create a Google Cloud project when prompted
4. Name your key (example: `adk-learning`) and click **Create key**
5. Copy the key once

### Store your API key

**For Google Colab:**
- Add your API key by clicking the Key icon on the left bar.
- Name your key (‘GOOGLE_API_KEY`) and add the API key from the previous step.
- Click Create key.
- Run the next cell, and it will prompt you to add the key via Secrets 🔑

**For Local Development:**
- Save your key to a `.env` file as: `GOOGLE_API_KEY=your-key-here`

> 🔐 Keep your API key private. Never commit `.env` files or paste your key in shared chats/slides.

Let's do this now! 👇


In [30]:
# Step 1: Load your API key from Colab Secrets (or local .env)

import os

# Try Colab Secrets first 
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    print("🔑 Using Colab Secrets")
except:
    # Fall back to .env 
    from dotenv import load_dotenv
    load_dotenv()
    GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
    print("🔑 Using .env file")

if not GOOGLE_API_KEY:
    print("❌ ERROR: GOOGLE_API_KEY not found")
    print("\n📌 For Colab Users:")
    print("   1. Click the 🔑 Secrets icon (left sidebar)")
    print("   2. Create new secret named 'GOOGLE_API_KEY'")
    print("   3. Paste your API key and save")
    print("\n📌 For Local Users:")
    print("   Add to .env: GOOGLE_API_KEY=your-key-here")
    print("\nGet a free key at: https://aistudio.google.com/app/api-keys")
else:
    # ✅ SET IT AS AN ENVIRONMENT VARIABLE (veli important!!!)
    os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
    print("✅ API Key loaded successfully and set in environment!")


🔑 Using .env file
✅ API Key loaded successfully and set in environment!


In [5]:
# Step 2: Import ADK components

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.genai import types

print("✅ ADK components imported successfully!")

✅ ADK components imported successfully!


In [6]:
# Step 3: Configure retry options (handles API rate limits gracefully)

retry_config = types.HttpRetryOptions(
    attempts=5,                              # Retry up to 5 times
    exp_base=7,                              # Exponential backoff
    initial_delay=1,                         # Start with 1 second delay
    http_status_codes=[429, 500, 503, 504]  # Retry on these errors
)

print("✅ Retry configuration set up!")
print("\n🎉 You're ready to build agents!")

✅ Retry configuration set up!

🎉 You're ready to build agents!


---

## 📋 How to Define an Agent

**Now let's understand the anatomy of an agent before we write tools.**

An agent in ADK is defined with a few key parameters. Let's break each one down.

---

### 🏗️ The Agent Structure

Every ADK agent has these core components:

```python
agent = LlmAgent(
    name="my_agent",                    # 1. Name (can be anything you like)
    model=Gemini(...),                  # 2. Model (the LLM)
    description="What this agent does", # 3. Description
    instruction="How to behave",        # 4. Instruction (MOST IMPORTANT)
    tools=[tool1, tool2, ...]           # 5. Tools (optional)
)
```



---

### 1️⃣ **Name: Identify Your Agent** 🤓

```python
name="helpful_assistant"
```

- Simple identifier for your agent
- Used in logs and tracing
- Should be descriptive but concise
- Examples: `"weather_bot"`, `"currency_converter"`, `"study_buddy"`

---

### 2️⃣ **Model: Choose the LLM Brain** 🧠 

```python
model=Gemini(
    model="gemini-2.5-flash-lite",
    retry_options=retry_config
)
```

This tells the agent which LLM to use for thinking:
- **`model`:** Which Gemini model
  - `"gemini-2.5-flash-lite"` - Fast, cheap, good for most things
  - `"gemini-2.5-flash"` - More capable, still fast
  - `"gemini-2.0-pro"` - Most powerful, slower
- **`retry_options`:** Handle API rate limits automatically

---

### 3️⃣ **Description: Tell Users What It Does** 📖

```python
description="A helpful assistant that answers questions about Python"
```

- Short explanation of the agent's purpose
- Helps users know what to ask it
- Helps other agents to know what it does
- Used in UI/logs
- Example: `"Weather agent that provides real-time weather info"`

---

### 4️⃣ **Instruction: The Agent's Personality & Behavior** ⭐ **MOST IMPORTANT**

```python
instruction="""You are a helpful Python expert.
Your goal is to answer questions about Python programming.
Be clear and provide examples when helpful.
If you don't know something, say so."""
```

This is **the most important part**. The instruction is like a prompt that tells the agent:
- **What role to play:** "You are a..."
- **What to do:** "Your job is to..."
- **How to behave:** "Be friendly, thorough, etc."
- **When to use tools:** "Use X tool for current info"

**Tips for good instructions:**
- Be specific about what the agent should do
- Mention tools by name if you want it to use them
- Tell it HOW to use tools: "Use search_tool when you need current info"
- Set expectations: "Provide detailed explanations"
- Mention constraints: "Don't make up information"

---

### 5️⃣ **Tools: Give It Hands to Take Action** (Optional) 🛠️

```python
tools=[get_weather, convert_currency, search_web]
```

- List of Python functions the agent can use
- The agent decides when to use each tool
- Each tool needs to be a function with proper format (we'll cover this next!)
- If no tools, the agent can only respond with text

---

### 📊 **Agent Definition at a Glance**

| Parameter | Purpose | Example | Required? |
|-----------|---------|---------|-----------|
| **name** | Agent identifier | `"weather_bot"` | Yes |
| **model** | Which LLM to use | `Gemini(model="gemini-2.5-flash-lite")` | Yes |
| **description** | What the agent does | `"Provides weather info"` | Yes |
| **instruction** | How to behave (CRUCIAL!) | `"You are a weather expert..."` | Yes |
| **tools** | Functions agent can use | `[get_weather, get_forecast]` | Optional |

---

### 💡 **Key Insight: Instruction is Where the Magic Happens**

The instruction is where you "program" the agent's behavior without code:

```python
# Example: Currency Converter Agent
instruction="""You are a currency conversion expert.
1. When asked to convert currency, use the exchange_rate tool
2. Also use the fee_calculator tool to find transaction fees
3. Always show the user the fee breakdown
4. Be precise with numbers"""
```

The agent reads this and knows:
- What tools to use
- When to use them
- How to format the response
- What matters (precision!)

This is MUCH more powerful than you'd think!

---

Now let's see this in action with actual code!

In [9]:
# Example 1: Creating a Simple Agent (No Tools Yet)

simple_agent = LlmAgent(
    name="friendly_assistant",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    description="A friendly helper that answers general questions",
    instruction="""You are a friendly and helpful assistant.
Your job is to answer questions clearly and concisely.
Be warm and encouraging in your tone."""
    # Note: No tools parameter - this agent can't use tools yet
)

print("✅ Agent created!")
print(f"Name: {simple_agent.name}")
print(f"Description: {simple_agent.description}")

✅ Agent created!
Name: friendly_assistant
Description: A friendly helper that answers general questions


In [29]:
# Let's test the simple agent (without tools)

# this runner allows us to execute the agent's logic without needing a full environment setup
simple_runner = InMemoryRunner(agent=simple_agent) 

print("Testing the simple agent...\n")
response = await simple_runner.run_debug(
    "What is the capital of France?"
)

Testing the simple agent...


 ### Created new session: debug_session_id

User > What is the capital of France?
friendly_assistant > The capital of France is Paris! It's a beautiful city known for its art, fashion, and delicious food. 🇫🇷


---

### 🎯 Notice: Same Agent, Different Instructions = Different Behavior

The agent's instruction completely changes how it responds. Let's create agents with different instructions and see the difference!

In [10]:
# Example 2: Expert Agent (Different Instruction)

expert_agent = LlmAgent(
    name="python_expert",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    description="Expert Python programmer who teaches with examples",
    instruction="""You are a world-class Python expert with 20 years of experience.
Your goal is to teach Python deeply and thoroughly.
Always provide:
1. Clear explanations
2. Working code examples
3. Best practices
4. Common pitfalls to avoid
Be thorough and educational."""
)

expert_runner = InMemoryRunner(agent=expert_agent)

print("Same question, but with EXPERT instruction:\n")
response = await expert_runner.run_debug(
    "What is the capital of France?"
)

print("\n💡 Notice: The expert agent gave a simple answer too.")
print("Why? Because the question doesn't require expertise.")


Same question, but with EXPERT instruction:


 ### Created new session: debug_session_id

User > What is the capital of France?
python_expert > That's a great question to start with! While I can certainly tell you the capital of France, my primary purpose is to teach you Python. I can explain concepts, show you how to write code, and help you understand Python best practices.

However, if you're curious, the capital of France is **Paris**.

Would you like to learn how to write a simple Python program that could tell you the capital of a country? We could even store this information in a data structure like a dictionary!

💡 Notice: The expert agent gave a simple answer too.
Why? Because the question doesn't require expertise.


---

### 🎨 Your Turn: Try Modifying an Agent

In the cell below, try changing:
- The **name** of the agent
- The **description** 
- The **instruction** (this has the biggest effect!)

Then run it and ask it a question. See how your changes affect how it responds!

In [ ]:
# TODO: Modify the agent below and run it!

my_custom_agent_def = LlmAgent(
    name="demo_agent",  # ← Change me!
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    description="A helpful demo agent",  # ← Change me!
    instruction="""You are a helpful assistant
    
    Ideas:
    - "You are a pirate who answers questions like a pirate"
    - "You are a helpful teacher for beginners"
    - "You are a funny comedian"
    - "You are a strict professional who is all business"
    
    Your instruction could be anything! Just be creative."""  # ← Change me!
)

custom_runner = InMemoryRunner(agent=my_custom_agent_def)

print("Testing YOUR custom agent...\n")
response = await custom_runner.run_debug(
    "What is the capital of France?"  # ← You can also change the question!
)

---

### ✅ What You've Learned About Agents

By now you understand:

✅ **Agent Structure**
- Every agent needs: name, model, description, instruction (+ optional tools)

✅ **The Instruction is Key**
- Instructions control agent behavior
- Same agent, different instruction = different behavior
- You "program" agents with natural language!

✅ **Parameters Do Different Things**
- **name:** Identifier (for logs/debugging)
- **model:** The LLM that powers thinking
- **description:** What the agent does
- **instruction:** HOW the agent behaves (MOST IMPORTANT)
- **tools:** What actions it can take

✅ **Agents Are Smart**
- They understand context
- They know when to use tools (via instruction)
- They synthesize information intelligently

---

### 🎯 Next: Tools

Now that you understand how agents are defined, let's learn how to write tools that agents can use.

Tools are the "hands" that let agents take action. Let's build them!

## 🔧 How to Write Tools for ADK

**Before we build an agent, let's understand how to write tools that agents can use.**

A tool in ADK is just a Python function, but it needs to follow some patterns so agents can understand and use it effectively.

---

### 📝 The Anatomy of an ADK Tool

Every good ADK tool has **4 key parts:**

```python
def my_tool(param1: str, param2: int) -> dict:
    """Clear description of what the tool does."""
    # 1. DOCSTRING (agent reads this to understand the tool)
    # 2. TYPE HINTS (agent needs to know what types to pass)
    # 3. LOGIC (do the actual work)
    # 4. RETURN DICT (always return {"status": "...", "data": ...})
    
    result = do_something(param1, param2)
    return {"status": "success", "result": result}
```

Step by step:
---

### 1️⃣ **Docstring: Your Tool's Manual**

The docstring is **critical** because the agent reads it to understand:
- What does this tool do?
- When should I use it?
- What parameters does it need?
- What will it return?

```python
def get_fun_fact(category: str) -> dict:
    """
    Returns a fun fact about a topic.
    
    This docstring tells the agent everything it needs to know:
    - PURPOSE: Returns fun facts
    - INPUT: Takes a category (string)
    - OUTPUT: Returns a dictionary with status and fact
    
    Args:
        category: The topic to get a fact about. 
                 Examples: "google", "python", "ai"
    
    Returns:
        Dictionary with:
        - "status": "success" or "error"
        - "fact" or "error_message": the content
    """
```

**Pro tip:** Write docstrings for HUMANS first (the agent will read them too).

---

### 2️⃣ **Type Hints: Data Types Matter**

Type hints tell the agent what kind of data to pass:

```python
def get_fun_fact(category: str) -> dict:
#                           ↑         ↑
#                      INPUT TYPE  OUTPUT TYPE
```

**Common types:**
- `str` - text string
- `int` - integer (whole number)
- `float` - decimal number
- `bool` - True/False
- `list` - array of items
- `dict` - dictionary/object

The agent uses these hints to format the data correctly.

---

### 3️⃣ **Function Logic: Do the Work**

This is just normal Python code. Do whatever you need:

```python
def get_fun_fact(category: str) -> dict:
    """Returns a fun fact about a topic."""
    
    # Create a data structure
    facts = {
        "google": "Google processes 8 billion searches/day",
        "python": "Python is 30+ years old",
    }
    
    # Look up the category
    fact = facts.get(category.lower())
    
    # Check if we found it
    if fact:
        return {"status": "success", "fact": fact}
    else:
        return {"status": "error", "error_message": f"Category '{category}' not found"}
```

---

### 4️⃣ **Return Structure: Always Use Status Pattern**

**CRITICAL:** Always return a dictionary with a "status" field:

```python
# SUCCESS case
return {
    "status": "success",
    "result": result_data,
    "extra_info": "optional metadata"
}

# ERROR case
return {
    "status": "error",
    "error_message": "Human-readable error description"
}
```

**Why this matters:**
- Agent can check: `if response["status"] == "error"`
- Agent knows if the tool worked or failed
- Agent can handle errors gracefully

---

### ✅ **Best Practices for ADK Tools**

| Practice | Why | Example |
|----------|-----|---------|
| **Clear docstring** | Agent reads it to understand tool | "Returns weather for a city" |
| **Type hints** | Agent knows what data to pass | `city: str` |
| **Single responsibility** | Tool does ONE thing well | Don't combine "weather AND forecast" |
| **Return status dict** | Agent knows if it worked | `{"status": "success", ...}` |
| **Handle errors gracefully** | Agent recovers from failures | Check if data exists before returning |
| **Validate inputs** | Tool doesn't crash on bad data | Check that `city` is not empty |
| **Clear parameter names** | Agent understands what to pass | Use `city` not `c` |

---

### 🚨 **Common Tool Mistakes to Avoid**

```python
# ❌ BAD: No docstring
def get_weather(city):
    # Agent has no idea what this does!
    return requests.get(f"api.weather.com?q={city}")

# ✅ GOOD: Clear docstring + types + status
def get_weather(city: str) -> dict:
    """Get weather for a city."""
    try:
        response = requests.get(f"api.weather.com?q={city}").json()
        return {"status": "success", "temp": response["temp"]}
    except Exception as e:
        return {"status": "error", "error_message": str(e)}
```

```python
# ❌ BAD: Returns raw data (agent can't tell if it worked)
def convert_currency(amount, from_curr, to_curr):
    return 100.50  # Is this success? Error? Unknown!

# ✅ GOOD: Always returns status dict
def convert_currency(amount: float, from_curr: str, to_curr: str) -> dict:
    """Convert amount from one currency to another."""
    rate = get_rate(from_curr, to_curr)
    if rate:
        return {"status": "success", "converted_amount": amount * rate}
    else:
        return {"status": "error", "error_message": f"No rate for {from_curr}/{to_curr}"}
```

---

### 🎯 **Tool Writing Checklist**

Before you give a tool to an agent, ask:

- [ ] Does it have a clear docstring?
- [ ] Does it have type hints on inputs?
- [ ] Does it return a dict with "status" field?
- [ ] Does it handle errors (try/except)?
- [ ] Does it validate inputs?
- [ ] Is it one focused task (not multiple things)?
- [ ] Can the agent understand what it does from the docstring?

---

Now let's see this in action! Here's our first tool:

In [11]:
# Step 1: Define a TOOL (just a Python function)

def get_fun_fact(category: str) -> dict:
    """
    Returns a fun fact about a topic.
    
    This is a TOOL that our agent can use.
    The docstring is important — the agent reads it to understand what the tool does.
    
    Args:
        category: The topic (e.g., "google", "python", "ai")
    
    Returns:
        Dictionary with status and fact
    """
    facts = {
        "google": "Google processes over 8 billion searches per day! 🔍",
        "python": "Python is 30+ years old and still the #1 language for AI 🐍",
        "ai": "Transformers (attention mechanism) changed AI forever in 2017 🤖",
        "gdg": "Google Developer Groups exist in dozens of cities worldwide! 🌍",
    }
    
    fact = facts.get(category.lower())
    if fact:
        return {"status": "success", "fact": fact}
    else:
        return {
            "status": "error",
            "error_message": f"Topic '{category}' not found. Try: google, python, ai, gdg"
        }

# Test the tool directly
print("Testing tool directly:")
print(get_fun_fact("google"))
print(get_fun_fact("python"))

Testing tool directly:
{'status': 'success', 'fact': 'Google processes over 8 billion searches per day! 🔍'}
{'status': 'success', 'fact': 'Python is 30+ years old and still the #1 language for AI 🐍'}


In [12]:
# Step 2: Create an AGENT with a tool

study_jam_agent = LlmAgent(
    name="study_jam_expert",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    description="A helpful agent for the GDG Study Jam",
    instruction="""You are a friendly expert at the GDG Study Jam.
    Your goal is to help participants learn about AI, Google, Python, and agents.
    When someone asks for information, use the get_fun_fact tool to find relevant facts.
    Be enthusiastic and encouraging!""",
    tools=[get_fun_fact]  # ← This is how we give the agent a tool
)

print("✅ Agent created!")
print(f"Agent name: {study_jam_agent.name}")
print(f"Tools available: get_fun_fact")

✅ Agent created!
Agent name: study_jam_expert
Tools available: get_fun_fact


In [ ]:
# Step 3: Create a RUNNER to execute the agent
runner = InMemoryRunner(agent=study_jam_agent)
print("✅ Runner created!")

✅ Runner created!


In [12]:
# Step 4: RUN your agent! 🚀

# Watch what happens:
# 1. Agent reads your question
# 2. Agent decides: "I need to use get_fun_fact tool"
# 3. Agent uses the tool to get information
# 4. Agent gives you a helpful answer

print("\n" + "="*60)
print("RUNNING YOUR AGENT...")
print("="*60 + "\n")

response = await runner.run_debug(
    "Tell me a fun fact about Google!"
)


RUNNING YOUR AGENT...


 ### Created new session: debug_session_id

User > Tell me a fun fact about Google!


study_jam_expert > I'd love to share a fun fact about Google with you! Did you know that Google processes over 8 billion searches *per day*? That's an incredible amount of information being accessed every single day! Keep up the great work in the Study Jam! 🚀


In [14]:
# Try another prompt with your agent!

response = await runner.run_debug(
    "I want to learn about Python. What's something cool about it?"
)


 ### Created new session: debug_session_id

User > I want to learn about Python. What's something cool about it?


study_jam_expert > That's awesome that you want to learn about Python! It's a fantastic language. Here's a cool fact for you:

Python is over 30 years old and is still the number one language for AI! How cool is that?! 🐍 Keep up the great work exploring it!


---

## 🎉 You Just Built an AI Agent!

Your agent:
1. **Read** your question
2. **Decided** it needed to use the fun_fact tool
3. **Executed** the tool
4. **Answered** you based on the result

That's the core of AI agents: **thinking → deciding → acting → answering**.

---

## ✍️ Section 3: Your Turn! Add Your Own Tool (20 min)

**Now you're going to customize the agent by adding YOUR OWN tool.**

### Option 1: University Facts Tool (Easy)
Create a tool that returns facts about your university.

### Option 2: Study Buddy Tool (Medium)
Create a tool that generates quiz questions or study tips.

### Option 3: Your Own Idea (Custom!)
Build any tool you want. Some ideas:
- A calculator that does specific math
- A motivational quote generator
- A event scheduler
- A campus resource finder

---

### 📝 Template: Copy this and fill in YOUR tool

In [ ]:
# TODO: Change the function name to something that describes your tool
def get_fun_fact(input_parameter: str) -> dict:
    """
    TODO: Write a description of what your tool does.
    The agent will read this to understand when to use it.
    
    Args:
        input_parameter: What does this parameter represent?
    
    Returns:
        Dictionary with "status" and the result
    """
    # TODO: Fill in your tool logic here
    # Create a dict of data or implement logic
    
    data = {
        "example_key_1": "example_value_1",
        "example_key_2": "example_value_2",
    }
    
    result = data.get(input_parameter.lower())
    if result:
        return {"status": "success", "result": result}
    else:
        return {"status": "error", "error_message": "Not found"}

# Test your tool
print("Testing your tool:")
print(get_fun_fact("example_key_1"))

### 🔧 Step 2: Create an agent with YOUR tool

Copy the code below and replace:
- `YOUR_TOOL_NAME` with your actual tool function name
- Update the agent name and description
- Update the instruction to describe what the agent should do with your tool

In [ ]:
# Create a new agent with YOUR tool

my_custom_agent = LlmAgent(
    name="my_agent",  # TODO: Change this to a cool name!
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    description="TODO: Describe what your agent does",
    instruction="""Use tools to help answer questions.
    Tell it WHEN and HOW to use your tool.
    Example: 'Use get_fun_fact to help answer questions about [topic]'""",
    tools=[get_fun_fact]  # TODO: Put your tool name here
)

# Create a runner for your custom agent
custom_runner = InMemoryRunner(agent=my_custom_agent)

print("✅ Your custom agent is ready!")

In [ ]:
# Test YOUR custom agent!

print("\n" + "="*60)
print("TESTING YOUR CUSTOM AGENT...")
print("="*60 + "\n")

# TODO: Write a prompt that asks your agent to use YOUR tool
response = await custom_runner.run_debug(
    "TODO: Write your test prompt here!"
)

## 🔥 Multi-Tool Agents: Taking It Further

Now that you understand tools, let's see what happens when an agent has **multiple tools** to choose from. Your agent becomes smarter. It can reason about which tool is best for each situation.


In [17]:
# Bonus Tool Example: A second tool for your agent

def get_study_tips(subject: str) -> dict:
    """
    Returns study tips for a subject.
    
    Args:
        subject: The subject to study (e.g., "python", "ai", "web")
    
    Returns:
        Dictionary with study tips
    """
    tips = {
        "python": "Build small projects! Start with loops, functions, then classes.",
        "ai": "Understand the fundamentals: vectors, matrices, probability.",
        "web": "Start with HTML/CSS, then JavaScript, then a framework.",
        "agents": "Understand tools first, then multi-agent systems.",
    }
    
    tip = tips.get(subject.lower())
    if tip:
        return {"status": "success", "tip": tip}
    else:
        return {"status": "error", "error_message": "Subject not found"}

print("Testing bonus tool:")
print(get_study_tips("python"))

Testing bonus tool:
{'status': 'success', 'tip': 'Build small projects! Start with loops, functions, then classes.'}


In [18]:
# Create an agent with MULTIPLE tools!

multi_tool_agent = LlmAgent(
    name="study_buddy",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    description="A study buddy that provides facts and tips",
    instruction="""You are a helpful study buddy.
    Use get_fun_fact to entertain with interesting facts.
    Use get_study_tips to provide study advice.
    Be encouraging!""",
    tools=[get_fun_fact, get_study_tips]  # ← Multiple tools!
)

bonus_runner = InMemoryRunner(agent=multi_tool_agent)
print("✅ Multi-tool agent created!")

✅ Multi-tool agent created!


In [20]:
# Test the multi-tool agent

print("\n" + "="*60)
print("TESTING MULTI-TOOL AGENT...")
print("="*60 + "\n")

response = await bonus_runner.run_debug(
    "Tell me a fun fact about AI and give me study tips for learning about agents!"
)


TESTING MULTI-TOOL AGENT...


 ### Continue session: debug_session_id

User > Tell me a fun fact about AI and give me study tips for learning about agents!
study_buddy > The Transformer architecture, with its attention mechanism, revolutionized AI in 2017! When studying agents, it's key to first get a solid grasp of the tools they use, and then move on to understanding how multi-agent systems work together. You've got this!


## 🤝 Multi-Agent Systems: Agents as Tools

**The Power of Specialization**

So far, we've built single agents—one agent with multiple tools. But what if you need **different agents with different expertise**?

### The Problem: One Agent Doing Everything

Imagine an agent that needs to:
1. Research a topic (find information, read articles)
2. Write a summary (clear, engaging writing)
3. Review for errors (grammar, factual accuracy)

You could give ONE agent all these tools:
```python
big_agent = LlmAgent(
    name="big_agent",
    tools=[search_tool, write_tool, review_tool],
    instruction="Do everything: research, write, and review"
)
```

❌ Problem: One agent trying to do everything is inefficient and confused.
- Instructions are long and complex
- The agent might mix up which mode it should be in
- It can't leverage specialized expertise

### The Solution: Multi-Agent Systems

Instead, create **specialized agents** and have them **collaborate**:

**Coordinator Agent** (the boss)
  → delegates to: Research Agent, Writer Agent, Review Agent

Each agent is an expert in ONE thing. The coordinator decides who should do what.

✅ Benefits:
- 🎯 Each agent has **clear, focused instructions**
- 🧠 Each agent becomes **expert** in its domain
- 📞 They communicate through the **Coordinator** agent
- 🔄 Complex workflows become **manageable**

### Real-World Examples

**Example 1: Customer Service**
| Role | Job | Example |
|------|-----|----------|
| 🔍 Lookup Agent | Find customer history | "Customer has 3 open tickets" |
| 💬 Support Agent | Write helpful response | "I see you're experiencing..." |
| ✅ QA Agent | Check quality & tone | Ensures response is professional |
| 🤝 Manager | Coordinate them | Orchestrates the workflow |

**Example 2: Content Creation**
| Role | Job |
|------|-----|
| 📚 Research Agent | Finds sources and facts |
| ✍️ Writer Agent | Creates engaging content |
| 📋 Editor Agent | Improves clarity and flow |

**Example 3: Code Review**
| Role | Job |
|------|-----|
| 🔎 Analysis Agent | Checks code structure |
| 🐛 Security Agent | Looks for vulnerabilities |
| 📝 Documentation Agent | Checks for comments |

### How It Works in ADK

The magic is `AgentTool`—it lets you **use an agent as a tool**:

```python
# Create specialized agents
researcher = LlmAgent(name="researcher", instruction="Find information", ...)
writer = LlmAgent(name="writer", instruction="Write clearly", ...)

# Use them as tools in a coordinator agent
coordinator = LlmAgent(
    name="coordinator",
    tools=[
        AgentTool(agent=researcher),  # ← Researcher is now a tool!
        AgentTool(agent=writer)       # ← Writer is now a tool!
    ]
)
```

When the coordinator needs research, it **calls the researcher agent**. When it needs writing, it **calls the writer agent**. This is **agent composition**—the foundation of large, intelligent systems.

Let's see it in action:


In [16]:
# Import AgentTool to use agents as tools
from google.adk.tools import AgentTool

# Create Agent 1: The Researcher
researcher_agent = LlmAgent(
    name="researcher",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    description="Specialist in finding information and facts",
    instruction="""You are a research specialist.
    Your job is to find relevant information and facts about topics.
    Be thorough and cite your sources when possible.
    Provide clear, well-organized information.""",
    tools=[get_fun_fact]  # Use the fun_fact tool to gather info
)

# Create Agent 2: The Writer
writer_agent = LlmAgent(
    name="writer",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    description="Specialist in writing engaging content",
    instruction="""You are a talented writer.
    Your job is to take information and write it in an engaging,
    clear, and well-structured way.
    Make the content interesting and easy to understand."""
)

# Create Agent 3: The Coordinator (uses the other agents as tools)
coordinator_agent = LlmAgent(
    name="content_coordinator",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    description="Coordinates research and writing",
    instruction="""You are a content coordinator.
    When someone asks you to create content:
    1. Ask the researcher agent to find information
    2. Ask the writer agent to write engaging content based on that info
    3. Combine their outputs into a final response
    
    Be clear about delegating tasks to the specialist agents.""",
    tools=[
        AgentTool(agent=researcher_agent),  # ← Agent as a tool!
        AgentTool(agent=writer_agent)       # ← Agent as a tool!
    ]
)

coordinator_runner = InMemoryRunner(agent=coordinator_agent)
print("✅ Multi-agent system ready!")

✅ Multi-agent system ready!


In [17]:
# Test the multi-agent system
print("="*60)
print("MULTI-AGENT SYSTEM IN ACTION")
print("="*60 + "\n")

response = await coordinator_runner.run_debug(
    "Write me an engaging article about Python for beginners. Start by researching interesting facts."
)

print("\n💡 What just happened:")
print("1. You asked the Coordinator")
print("2. Coordinator asked the Researcher agent for facts")
print("3. Researcher used the fact-finding tool")
print("4. Coordinator then asked the Writer to write engaging content")
print("5. All three agents worked together!")
print("\nThat's multi-agent systems. Agents delegating to other agents!")

MULTI-AGENT SYSTEM IN ACTION


 ### Created new session: debug_session_id

User > Write me an engaging article about Python for beginners. Start by researching interesting facts.
content_coordinator > ## Python: Your Gateway to the Future (That's Been Around for 30+ Years!)

Ever wondered how your favorite apps, the smart assistants on your phone, or even the algorithms that recommend your next binge-watch work? Chances are, a little bit of Python magic is involved. And the best part? This incredibly powerful and popular programming language is remarkably welcoming to newcomers, even though it's been crafting digital wonders for over three decades!

That's right, Python isn't some shiny new fad. It first graced the world with its elegant syntax back in the early 1990s. Since then, it's evolved, grown, and become a cornerstone of the tech industry, consistently ranking among the top programming languages globally.

### Why the Fuss About Python? Let's Break It Down:

**1. The "Beginner-

## 💾 Sessions: Agent Memory Across Turns

**The Problem:** Without sessions, agents have **no memory**. Every message is treated as brand new.

```
User Turn 1: "Hi! I'm learning Python."
Agent: "Cool! What are you learning?"

User Turn 2: "Can you remind me what I told you?"
Agent: "I don't know what you told me"  ❌ Memory lost!
```

**The Solution:** Use **Sessions** to store conversation history and state.

### 🔑 Key Concept: InMemoryRunner vs Runner

To enable sessions, you need to use **`Runner`** instead of **`InMemoryRunner`**:

| Feature | InMemoryRunner | Runner |
|---------|---|---|
| **Session Support** | ❌ No | ✅ Yes |
| **Memory** | ❌ None | ✅ Persistent |
| **Best For** | Single Q&A | Multi-turn chats |
| **Method** | `run_debug()` | `run_async()` |

### Why Runner for Stateful Conversations?

**InMemoryRunner** is simple but stateless:
```python
runner = InMemoryRunner(agent=my_agent)
response = await runner.run_debug("Hello")  # No memory!
```

**Runner** connects to sessions and remembers:
```python
runner = Runner(
    agent=my_agent,
    app_name="my_app",
    session_service=session_service  # ← The magic!
)

# Message 1 - Stored in session
async for event in runner.run_async(
    user_id="alice",
    session_id="session_123",
    new_message=types.Content(parts=[types.Part(text="My goal is Python")])
):
    pass  # Automatically saved!

# Message 2 - Retrieved from session
async for event in runner.run_async(
    user_id="alice",
    session_id="session_123",  # Same session!
    new_message=types.Content(parts=[types.Part(text="What was my goal?")])
):
    # ✅ Agent remembers: "Your goal is Python"
```

### How Sessions Work

1. **Create Session Service** → Where to store data
2. **Create a Session** → Open a conversation notebook
3. **Create Runner with Session Service** → Connect to storage
4. **Pass session_id** → Tell runner which conversation to use
5. **Agent remembers automatically** → No extra code!

Let's build a study buddy that remembers! 👇

In [ ]:
# Create a study buddy agent that remembers your goals
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types
import uuid

study_buddy_agent = LlmAgent(
    name="study_buddy",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    description="A study buddy that remembers your learning goals",
    instruction="""You are a supportive study buddy.
    Your role is to:
    1. Remember what the student tells you about their goals
    2. Provide encouragement and study tips
    3. Reference their goals in future messages
    
    Be friendly, encouraging, and helpful.
    Show that you remember what they've told you by referencing it.""",
    tools=[get_study_tips]  # Reuse the bonus tool from earlier
)

# Create a session service (for memory!)
session_service = InMemorySessionService()

# Create a session for this conversation
study_session_id = str(uuid.uuid4()) # Unique session ID to track this conversation's memory
await session_service.create_session(
    app_name="gdg_study_jam",
    user_id="participant",
    session_id=study_session_id
)

# Create a STATEFUL runner using Runner (not InMemoryRunner)
session_runner = Runner( # use Runner for stateful conversations!
    agent=study_buddy_agent,
    app_name="gdg_study_jam",
    session_service=session_service
)

print("✅ Study buddy agent ready for multi-turn conversations!")
print(f"   Session ID: {study_session_id}")
print("   It will REMEMBER your goals across multiple messages!\n")

# Demonstrate multi-turn conversation with memory
print("=" * 60)
print("TESTING STUDY BUDDY WITH MEMORY")
print("=" * 60)

# Message 1: Tell your goal
print("\n📝 Turn 1: You tell your study goal")
async for event in session_runner.run_async(
    user_id="participant",
    session_id=study_session_id,
    new_message=types.Content(parts=[types.Part(text="Hi my name is [Participant]. I want to learn about AI agents and build my first one!")])
):
    if event.is_final_response() and event.content and event.content.parts:
        print(f"Study Buddy: {event.content.parts[0].text}")

# Message 2: Agent REMEMBERS your goal!
print("\n📝 Turn 2: You ask for study tips")
async for event in session_runner.run_async(
    user_id="participant",
    session_id=study_session_id,
    new_message=types.Content(parts=[types.Part(text="What are the best resources to learn about agents?")])
):
    if event.is_final_response() and event.content and event.content.parts:
        print(f"Study Buddy: {event.content.parts[0].text}")

# Message 3: Agent still remembers!
print("\n📝 Turn 3: Verify it remembers your goal")
async for event in session_runner.run_async(
    user_id="participant",
    session_id=study_session_id,
    new_message=types.Content(parts=[types.Part(text="Can you remind me what I said I wanted to learn? Also what is my name?")])
):
    if event.is_final_response() and event.content and event.content.parts:
        print(f"Study Buddy: {event.content.parts[0].text}")

print("\n✨ See? The study buddy REMEMBERS your goal across all messages!")
print("   This is what stateful runners do - they connect to sessions!\n")




✅ Study buddy agent ready for multi-turn conversations!
   Session ID: aa32f812-c438-4b4a-9ed2-7be193f21f09
   It will REMEMBER your goals across multiple messages!

TESTING STUDY BUDDY WITH MEMORY

📝 Turn 1: You tell your study goal
Study Buddy: Hi [Participant]! It's great to meet you. Learning about AI agents and building your first one sounds like a fantastic goal! I'm excited to help you on your journey. We'll work together to make sure you achieve this. 


📝 Turn 2: You ask for study tips
Study Buddy: That's a great question! Since you're aiming to build your first AI agent, diving into the right resources will definitely set you up for success.

Would you like me to find some general study tips for learning about AI agents, or are you interested in resources for a specific subject, like programming languages or frameworks commonly used for building agents? Let me know what you think would be most helpful right now!


📝 Turn 3: Verify it remembers your goal
Study Buddy: You told 

In [27]:
# View the complete session history
print("\n" + "="*60)
print("📚 FULL SESSION HISTORY")
print("="*60 + "\n")

# Get the session from the service
session = await session_service.get_session(
    app_name="gdg_study_jam",
    user_id="participant",
    session_id=study_session_id
)

# Display session info
print(f"Session ID: {session.id}")
print(f"User: {session.user_id}")
print(f"App: {session.app_name}")
print(f"Total events: {len(session.events)}")
print()

# Print all conversation events
for i, event in enumerate(session.events, 1):
    print(f"📝 Event {i}:")
    if event.content and event.content.parts:
        for part in event.content.parts:
            if part.text:
                print(f"   {part.text}")

print("\n" + "="*60)
print("💡 Key Insight: The session stores EVERYTHING!")
print("   All messages, all events, all state.")
print("   This is how agents maintain context across turns.")
print("="*60)



📚 FULL SESSION HISTORY

Session ID: aa32f812-c438-4b4a-9ed2-7be193f21f09
User: participant
App: gdg_study_jam
Total events: 6

📝 Event 1:
   Hi my name is [Participant]. I want to learn about AI agents and build my first one!
📝 Event 2:
   Hi [Participant]! It's great to meet you. Learning about AI agents and building your first one sounds like a fantastic goal! I'm excited to help you on your journey. We'll work together to make sure you achieve this. 

📝 Event 3:
   What are the best resources to learn about agents?
📝 Event 4:
   That's a great question! Since you're aiming to build your first AI agent, diving into the right resources will definitely set you up for success.

Would you like me to find some general study tips for learning about AI agents, or are you interested in resources for a specific subject, like programming languages or frameworks commonly used for building agents? Let me know what you think would be most helpful right now!

📝 Event 5:
   Can you remind me what 

### 🎁 BONUS: Real-World Example - Currency Converter

Here's a practical example combining multiple tools:


In [ ]:
# BONUS 1: Build a Currency Converter with Multiple Tools
# This agent uses TWO tools to convert currency with fees

# Tool 1: Get exchange rates
def get_exchange_rate(from_currency: str, to_currency: str) -> dict:
    """
    Get the exchange rate between two currencies.
    
    Args:
        from_currency: Source currency code (USD, EUR, JPY, etc.)
        to_currency: Target currency code
    
    Returns:
        Dictionary with status and exchange rate
    """
    rates = {
        ("usd", "eur"): 0.93,
        ("usd", "jpy"): 157.50,
        ("usd", "gbp"): 0.79,
        ("eur", "u222sd"): 1.08,
        ("gbp", "usd"): 1.27,
    }
    
    key = (from_currency.lower(), to_currency.lower())
    rate = rates.get(key)
    
    if rate:
        return {"status": "success", "rate": rate}
    else:
        return {
            "status": "error",
            "error_message": f"Exchange rate not found for {from_currency}/{to_currency}"
        }

# Tool 2: Get transaction fees
def get_transaction_fee(payment_method: str) -> dict:
    """
    Get the transaction fee percentage for a payment method.
    
    Args:
        payment_method: Type of payment (credit_card, bank_transfer, etc.)
    
    Returns:
        Dictionary with status and fee percentage
    """
    fees = {
        "credit_card": 0.025,  # 2.5%
        "debit_card": 0.015,   # 1.5%
        "bank_transfer": 0.005, # 0.5%
        "paypal": 0.035,        # 3.5%
    }
    
    fee = fees.get(payment_method.lower())
    
    if fee is not None:
        return {"status": "success", "fee_percentage": fee}
    else:
        return {
            "status": "error",
            "error_message": f"Payment method '{payment_method}' not supported"
        }

# Test the tools
print("Exchange Rate Test:")
print(get_exchange_rate("USD", "EUR"))
print("\nFee Test:")
print(get_transaction_fee("credit_card"))

In [ ]:
# Create a currency converter agent with BOTH tools
converter_agent = LlmAgent(
    name="currency_converter",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    description="A currency conversion expert",
    instruction="""You are a currency conversion expert.
    
    For currency conversion requests:
    1. Use get_exchange_rate() to find the conversion rate
    2. Use get_transaction_fee() to find the fee for the payment method
    3. Check the "status" field - if it says "error", explain the problem
    4. Calculate: 
       - Fee amount = original amount × fee percentage
       - Amount after fee = original amount - fee amount
       - Final amount = amount after fee × exchange rate
    5. Provide a clear breakdown showing:
       - The fee percentage and fee amount
       - The exchange rate used
       - The final converted amount
    
    Always be precise with numbers.""",
    tools=[get_exchange_rate, get_transaction_fee]  # Both tools!
)

converter_runner = InMemoryRunner(agent=converter_agent)
print("✅ Currency converter agent created with 2 tools!")

In [ ]:
# Test the converter agent with multiple tools
print("="*60)
print("TESTING MULTI-TOOL AGENT")
print("="*60 + "\n")

response = await converter_runner.run_debug(
    "Convert 1000 USD to EUR using a credit card. Show me the breakdown."
)

---

## 🎓 Extended Resources: Learn More

Want to dive deeper? Here are some advanced topics and resources to explore at home.


### 📚 RAG (Retrieval-Augmented Generation)

RAG is a pattern where your agent can **retrieve information** from a knowledge base before responding. Instead of relying only on the LLM's training data, it accesses up-to-date documents.

**Example pattern:**
1. User asks a question
2. Agent uses a retrieval tool to find relevant documents
3. Agent reasons over those documents
4. Agent provides an informed answer

Libraries like **LangChain** and **LlamaIndex** provide RAG abstractions. In ADK, you'd typically implement this as a tool that queries your knowledge base.


### 🔗 Further Resources

- **Google ADK Documentation**: https://google.github.io/adk-docs/
- ***5-Day AI Agents Intensive Course with Google***: https://www.kaggle.com/learn-guide/5-day-agents (A more comprehensive version of what was discussed today)
- **Agent Design Patterns**: Study how prompt engineering shapes agent behavior
- **Tool Library Design**: Build reusable tool collections for common tasks
- **Evaluation**: How to measure agent performance and reliability.
- **Deployment**: Taking your agent from notebook to production (FastAPI, Cloud Run, etc.) 


### Created by: Miguel Varilla (github: https://github.com/miggle711)